In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import json


try:
    nombre_carpeta = dbutils.widgets.get("folder_name")
    cod_indicador = dbutils.widgets.get("indicador_code")
except Exception:
    # Valores por defecto si no están definidos
    #nombre_carpeta = "pbi"
    #cod_indicador = "NY.GDP.MKTP.CD"
    print(f"✅ Widgets listos. Procesando: {nombre_carpeta}")


class BronzeIngestor:
    def __init__(self, indicador_nombre, esquema):
        self.indicador = indicador_nombre
        self.esquema = esquema
        # Tabla en Unity Catalog (Puntos)
        self.tabla_destino = f"socioeconomics.bronze.bronze_{self.indicador}"
        # Carpeta para que Spark "anote" qué ya leyó (Barras)
        self.checkpoint_path = f"/Volumes/socioeconomics/landing/checkpoints/{self.indicador}"

    def ejecutar_streaming(self, ruta_carpeta_landing):
        print(f"📡 Iniciando vigilancia en: {ruta_carpeta_landing}")
        
        # 1. Configuramos el Auto Loader
        df_stream = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", f"{self.checkpoint_path}/schema")
            .option("multiLine", "true") # Para leer el JSON completo del World Bank
            .schema(self.esquema)
            .load(ruta_carpeta_landing))

        # 2. Agregamos auditoría básica
        df_final = df_stream.withColumn("_ingestado_el", F.current_timestamp())

        # 3. Guardamos de forma incremental
        # trigger(availableNow=True) procesa todo lo que hay y se detiene (ideal para batches)
        query = (df_final.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", f"{self.checkpoint_path}/data")
            .trigger(availableNow=True)
            .toTable(self.tabla_destino))
        
        query.awaitTermination()
        print(f"✅ Ingesta terminada en {self.tabla_destino}")

In [0]:
# 1. Widgets e Indicador
indicador = dbutils.widgets.get("folder_name") 

# 2. Esquema
%run ./schema_world_bank.ipynb

# 3. Ruta segura (Usando un Volume que ya sabemos que existe)
# Asumiendo que 'world_bank' es tu Volume donde caen los JSON
ruta_checkpoints = f"/Volumes/socioeconomics/landing/world_bank/checkpoints/{indicador}"

# 4. Crear las carpetas internas (Esto sí funciona si el Volume existe)
dbutils.fs.mkdirs(f"{ruta_checkpoints}/data")
dbutils.fs.mkdirs(f"{ruta_checkpoints}/schema")

# 5. Ejecutar Ingestor
# Asegurate que en el __init__ de tu clase uses esta misma lógica de ruta
ingestor = BronzeIngestor(indicador, SCHEMA_WORLD_BANK)
ingestor.checkpoint_path = ruta_checkpoints # Forzamos la ruta segura
ingestor.ejecutar_streaming("/Volumes/socioeconomics/landing/world_bank/")